# Linear maps of the plane

*maths · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/linear_maps_2d)

A $2\times 2$ matrix moves every point of the plane, and one einsum moves a whole grid, the unit circle and the unit square for every animation frame at once. Watch the plane morph from the identity to $A$: eigenvectors stay on their own lines, $|\det A|$ scales areas, a negative determinant flips the plane over, and the circle becomes an ellipse whose semi-axes are the singular values.

**Vector form**

$$\mathbf y = A\,\mathbf x,\qquad M(t) = (1 - t)\,I + t\,A,\qquad A\,\mathbf v = \lambda\,\mathbf v,\qquad A = U\,\Sigma\,V^{\mathsf T}$$

**Index (tensor) form**

$$Y_{tki} = M_{tij}\,P_{kj},\qquad M_{tij} = (1 - t)\,\delta_{ij} + t\,A_{ij},\qquad \det M = \varepsilon_{ij}\,M_{i1}\,M_{j2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{ij}$ | `matrices(choice, custom, dev)` | (S, 2, 2) | the matrices to show: stretch, shear, rotation, reflection and a flip, or your own |
| $P_{kj}$ | `grid_lines(...), unit_circle(...), unit_square(...)` | (…, 2) | points of the plane: grid lines, the unit circle, the unit square |
| $M_{tij}$ | `interpolate(A, t)` | (T, 2, 2) | straight path from the identity $\delta_{ij}$ to $A$, one matrix per frame |
| $Y_{tki} = M_{tij} P_{kj}$ | `einsum('tij,...j->t...i', M, P)` | (T, …, 2) | repeated $j$ ⇒ summed: every point moved by every frame's matrix at once |
| $\varepsilon_{ij}$ | `levi_civita(dev)` | (2, 2) | $\varepsilon_{12} = 1$, $\varepsilon_{21} = -1$: the 2-D Levi-Civita symbol |
| $\det M$ | `einsum('ij,...i,...j->...', eps, M[..., 0], M[..., 1])` | (T,) | repeated $i, j$ ⇒ summed: signed area of the image of the unit square |
| $U,\ \sigma$ | `torch.linalg.svd(M)` | (T, 2, 2), (T, 2) | the circle's image is an ellipse with semi-axes $\sigma_1 \mathbf u_1$ and $\sigma_2 \mathbf u_2$ |
| $\lambda,\ \mathbf v$ | `torch.linalg.eig(A)` | (2,), (2, 2) | eigenvalues and eigenvectors; complex for a rotation, so no line stays put |

The columns of $A$ are where the basis vectors $\mathbf e_1$ (red arrow) and $\mathbf e_2$ (green arrow) land, and every other point follows by linearity, so straight grid lines stay straight and parallel. Along the path $M(t) = (1-t)I + tA$ an eigenvector $\mathbf v$ of $A$ is also an eigenvector of $M(t)$ with eigenvalue $1 - t + t\lambda$, so the orange dots slide along their dashed lines while everything else turns. The unit square (yellow) becomes a parallelogram of area $|\det A|$; it turns red when $\det < 0$, when $\mathbf e_1 \to \mathbf e_2$ goes clockwise and the plane is a mirror image. A reflection must pass through $\det = 0$ on the way, where the plane is squashed onto a line. Rotations and reflections keep lengths, so $\sigma_1 = \sigma_2 = 1$; a shear keeps area ($\det = 1$) but not shape. In general the circle goes to an ellipse with semi-axes $\sigma_1 \ge \sigma_2$, the singular values, and $|\det A| = \sigma_1\sigma_2$. Products multiply: $\det(AB) = \det A\,\det B$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/linear_maps_2d/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import apply, det


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    custom = [params["a11"], params["a12"], params["a21"], params["a22"]]
    a, captions = tensors.matrices(params["matrix"], custom, dev)  # (S, 2, 2)
    eps = tensors.levi_civita(dev)
    lines = tensors.grid_lines(params["grid_lines"], 4.0, dev)  # (L, 41, 2), covering [−4, 4]²
    circle = tensors.unit_circle(200, dev)  # (200, 2)
    square = tensors.unit_square(dev)  # (5, 2)

    per = 70 // len(a)  # at most 70 frames in all
    moving = per * 3 // 4  # 3/4 of them walk from I to A, the rest rest on A
    m, which, t = solve.path(a, frames=moving, hold=per - moving)  # (T, 2, 2)
    y_lines, y_circle, y_square = apply(m, lines), apply(m, circle), apply(m, square)  # (T, …, 2)
    print(f"{len(a)} matrices, {m.shape[0]} frames: one einsum maps {lines.shape[0] * lines.shape[1]} grid points")

    u, sigma = solve.singular(m)  # (T, 2, 2), (T, 2)
    d = det(m, eps)  # (T,)
    axes = solve.circle_axes(y_circle)  # (T, 2)
    eig = []
    for s, (a_s, caption) in enumerate(zip(a, captions)):
        lam, vec = solve.real_eigenvectors(a_s)
        eig.append(vec)
        end = (which == s).nonzero()[-1].item()  # last frame of this matrix: M = A
        rows = ", ".join(f"[{r[0]:+.2f} {r[1]:+.2f}]" for r in a_s.tolist())
        print(f"A = [{rows}]  {caption}")
        print(f"   det A = {d[end]:+.3f} (area ×{abs(d[end]):.3f}{', mirrored' if d[end] < 0 else ''})")
        print(
            f"   singular values {sigma[end, 0]:.3f}, {sigma[end, 1]:.3f};  circle image axes "
            f"{axes[end, 0]:.3f}, {axes[end, 1]:.3f}"
        )
        if lam.numel():
            v_moved = a_s @ vec  # A v = λ v: the image lies on the same line
            print(
                f"   eigenvalues {', '.join(f'{v:+.3f}' for v in lam.tolist())};  "
                f"|A v − λ v| = {(v_moved - vec * lam).abs().max():.1e}"
            )
        else:
            print("   no real eigenvalues: no direction stays on its own line")
    return plot.animate(
        m, which, t, y_lines, y_circle, y_square, u, sigma, d, eig, captions, out_dir / "linear_maps_2d.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/linear_maps_2d/equation.py
r"""Maths → tensor translation.

A 2×2 matrix A moves every point of the plane, y_i = A_{ij} x_j (j summed). Stack many points in P_{kj} (k = point)
and many matrices in M_{tij} (t = animation frame) and every image of every point is one einsum,

    Y_{tki} = M_{tij}\,P_{kj}            (j summed).

The animation walks from the identity to A along M(t) = (1 − t) I + t A. Because M(t) is a mix of I and A, an
eigenvector of A (A v = λ v) is also an eigenvector of every M(t): it only slides along its own line.

The determinant, the signed area of the image of the unit square, is

    \det M = \varepsilon_{ij}\,M_{i1}\,M_{j2}            (i, j summed),

with ε the 2-D Levi-Civita symbol. |det| scales every area and a negative det flips orientation (a mirror image).
"""

import torch


def apply(m: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    r"""Y_{t…i} = M_{tij} P_{…j}   (j summed).   M (T, 2, 2), P (…, 2) → (T, …, 2)"""
    return torch.einsum("tij,...j->t...i", m, p)


def interpolate(a: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    r"""M_{tij} = (1 − t) δ_{ij} + t A_{ij}: the straight path from the identity to A.   t (T,) → (T, 2, 2)"""
    eye = torch.eye(2, dtype=a.dtype, device=a.device)
    return (1 - t)[:, None, None] * eye + t[:, None, None] * a


def det(m: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""det M = ε_{ij} M_{i1} M_{j2}   (i, j summed): the signed area spanned by the two columns.   (…, 2, 2) → (…)"""
    return torch.einsum("ij,...i,...j->...", eps, m[..., :, 0], m[..., :, 1])


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/linear_maps_2d/tensors.py
"""Build the concrete tensors: the example matrices, the grid lines, the unit circle and the unit square."""

import math

import torch

c, s = math.cos(math.radians(60)), math.sin(math.radians(60))
# name: (matrix A, caption). Every matrix is shown as a path from the identity to A.
MATRICES = {
    "stretch": ([[2.0, 1.0], [0.5, 1.5]], "eigenvalues 2.5 and 1: two lines stay put"),
    "shear": ([[1.0, 1.0], [0.0, 1.0]], "shear: area kept, one eigen-line"),
    "rotation": ([[c, -s], [s, c]], "rotation by 60°: no real eigenvectors"),
    "reflection": ([[-0.5, s], [s, 0.5]], "reflection in the 60° line: det = −1"),
    "flip": ([[0.5, 1.2], [1.0, 0.2]], "det < 0: orientation flips"),
}
TOUR = ["stretch", "shear", "rotation", "reflection", "flip"]


def matrices(choice: str, custom: list[float], device: torch.device) -> tuple[torch.Tensor, list[str]]:
    """The matrices to show, A_{sij} of shape (S, 2, 2), and a caption for each."""
    if choice == "tour":
        names = TOUR
    elif choice == "custom":
        a = torch.tensor(custom, dtype=torch.float64, device=device).reshape(1, 2, 2)
        return a, [f"custom A = [[{custom[0]:g}, {custom[1]:g}], [{custom[2]:g}, {custom[3]:g}]]"]
    else:
        names = [choice]
    a = torch.tensor([MATRICES[n][0] for n in names], dtype=torch.float64, device=device)
    return a, [MATRICES[n][1] for n in names]


def grid_lines(n_lines: int, half: float, device: torch.device, samples: int = 41) -> torch.Tensor:
    """Points along n_lines vertical and n_lines horizontal lines covering [−half, half]²: (2·n_lines, samples, 2)."""
    ticks = torch.linspace(-half, half, n_lines, dtype=torch.float64, device=device)  # where the lines sit
    along = torch.linspace(-half, half, samples, dtype=torch.float64, device=device)  # points on each line
    vertical = torch.stack([ticks[:, None].expand(-1, samples), along[None, :].expand(n_lines, -1)], dim=-1)
    horizontal = vertical.flip(-1)  # swap x and y
    return torch.cat([vertical, horizontal])


def unit_circle(n: int, device: torch.device) -> torch.Tensor:
    """Points (cos θ, sin θ) on the unit circle: (n, 2)."""
    theta = torch.linspace(0, 2 * math.pi, n, dtype=torch.float64, device=device)
    return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)


def unit_square(device: torch.device) -> torch.Tensor:
    """Corners of the unit square, counter-clockwise and closed: (5, 2). Its image has area |det A|."""
    return torch.tensor([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=torch.float64, device=device)


def levi_civita(device: torch.device) -> torch.Tensor:
    """ε_{ij}: ε_12 = 1, ε_21 = −1, 0 on the diagonal: (2, 2)."""
    return torch.tensor([[0.0, 1.0], [-1.0, 0.0]], dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/linear_maps_2d/solve.py
"""Build the animation path through all matrices and work out det, singular values and eigenvectors along it."""

import torch

from .equation import interpolate


def path(a: torch.Tensor, frames: int, hold: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """For each matrix: `frames` steps from I to A, then `hold` frames resting on A.

    a (S, 2, 2) → M (T, 2, 2), the matrix index of each frame (T,), and t of each frame (T,), T = S·(frames + hold).
    """
    move = torch.linspace(0, 1, frames, dtype=a.dtype, device=a.device)
    t = torch.cat([move, torch.ones(hold, dtype=a.dtype, device=a.device)])
    m = torch.cat([interpolate(a_s, t) for a_s in a])
    which = torch.arange(len(a), device=a.device).repeat_interleave(len(t))
    return m, which, t.repeat(len(a))


def singular(m: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """M = U Σ Vᵀ: the unit circle goes to an ellipse with semi-axes σ₁ u₁ and σ₂ u₂.

    (T, 2, 2) → U (T, 2, 2), σ (T, 2)
    """
    u, sigma, _ = torch.linalg.svd(m)
    return u, sigma


def real_eigenvectors(a: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigenvalues λ (k,) and unit eigenvectors in the columns (2, k) of one 2×2 matrix; k = 0 if they are complex."""
    lam, vec = torch.linalg.eig(a)
    if (lam.imag.abs() > 1e-9).any():  # complex pair: every direction turns (a rotation)
        return lam.real[:0], vec.real[:, :0]
    return lam.real, vec.real


def circle_axes(y_circle: torch.Tensor) -> torch.Tensor:
    """Longest and shortest distance from the origin to the image of the unit circle: (…, K, 2) → (…, 2)."""
    r = y_circle.norm(dim=-1)
    return torch.stack([r.amax(dim=-1), r.amin(dim=-1)], dim=-1)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/linear_maps_2d/plot.py
"""Animate the plane being transformed (left) with det and the singular values along the path (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

LIM = 3.2


def animate(
    m: torch.Tensor,
    which: torch.Tensor,
    t: torch.Tensor,
    y_lines: torch.Tensor,
    y_circle: torch.Tensor,
    y_square: torch.Tensor,
    u: torch.Tensor,
    sigma: torch.Tensor,
    d: torch.Tensor,
    eig: list[torch.Tensor],
    captions: list[str],
    path: Path,
) -> Path:
    """m (T, 2, 2); which, t (T,); y_lines (T, L, S, 2); y_circle (T, K, 2); y_square (T, 5, 2); u (T, 2, 2);
    sigma (T, 2); d (T,); eig[s] = eigenvectors of matrix s in the columns (2, k)."""
    m, which, t, y_lines, y_circle, y_square, u, sigma, d = (
        x.cpu().numpy() for x in (m, which, t, y_lines, y_circle, y_square, u, sigma, d)
    )
    eig = [v.cpu().numpy() for v in eig]
    frames_all = np.arange(len(t))

    fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(8, 4.1), gridspec_kw={"width_ratios": [1.15, 1]})
    ax.set_xlim(-LIM, LIM)
    ax.set_ylim(-LIM, LIM)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.axhline(0, color="#d1d5db", lw=0.8)
    ax.axvline(0, color="#d1d5db", lw=0.8)
    grid = [ax.plot([], [], color="#93c5fd", lw=0.7)[0] for _ in range(y_lines.shape[1])]
    eig_lines = [ax.plot([], [], color="#f59e0b", lw=1.2, ls="--")[0] for _ in range(2)]
    eig_dots = [ax.plot([], [], "o", color="#f59e0b", ms=6, mec="#b45309")[0] for _ in range(2)]
    (square,) = ax.fill([], [], color="#fde68a", lw=1.5, ec="#b45309", zorder=1)
    (circle,) = ax.plot([], [], color="#2563eb", lw=2)
    svd_axes = [ax.plot([], [], color="#9333ea", lw=1.8, ls=ls)[0] for ls in ("-", "--")]
    arrows: list = []
    caption = ax.set_title("two lines\nof caption", fontsize=10)  # placeholder so tight_layout leaves room

    ax_r.plot(frames_all, d, color="#374151", lw=1.6, label="det M")
    ax_r.plot(frames_all, sigma[:, 0], color="#9333ea", lw=1.2, label="σ₁ (long axis)")
    ax_r.plot(frames_all, sigma[:, 1], color="#9333ea", lw=1.2, ls="--", label="σ₂ (short axis)")
    ax_r.axhline(0, color="#9ca3af", lw=0.8)
    for s in range(len(captions)):  # shade every other matrix
        if s % 2:
            span = frames_all[which == s]
            ax_r.axvspan(span[0], span[-1], color="#f3f4f6", lw=0)
    marker = ax_r.axvline(0, color="#dc2626", lw=1.2)
    ax_r.set_xlim(0, len(t) - 1)
    ax_r.set_xlabel("frame (each block: I → A)")
    ax_r.set_title("|det| = area factor = σ₁σ₂", fontsize=10)
    ax_r.legend(fontsize=8, loc="lower left")

    def arrow(start, vec, color):
        return ax.annotate(
            "", xy=start + vec, xytext=start, arrowprops={"arrowstyle": "-|>", "color": color, "lw": 1.8}
        )

    def update(k: int):
        s = which[k]
        for line, pts in zip(grid, y_lines[k]):
            line.set_data(pts[:, 0], pts[:, 1])
        circle.set_data(y_circle[k, :, 0], y_circle[k, :, 1])
        square.set_xy(y_square[k])
        square.set_facecolor("#fecaca" if d[k] < 0 else "#fde68a")
        vecs = eig[s]
        for j in range(2):
            if j < vecs.shape[1]:
                v = vecs[:, j]
                eig_lines[j].set_data([-5 * v[0], 5 * v[0]], [-5 * v[1], 5 * v[1]])
                mv = m[k] @ v  # M(t) v stays on the line through v
                eig_dots[j].set_data([mv[0]], [mv[1]])
            else:
                eig_lines[j].set_data([], [])
                eig_dots[j].set_data([], [])
        while arrows:
            arrows.pop().remove()
        origin = np.zeros(2)
        arrows.append(arrow(origin, m[k][:, 0], "#dc2626"))  # image of e₁ (first column)
        arrows.append(arrow(origin, m[k][:, 1], "#16a34a"))  # image of e₂ (second column)
        for j in range(2):  # the ellipse's axes run from −σ_j u_j to +σ_j u_j
            end = sigma[k, j] * u[k][:, j]
            svd_axes[j].set_data([-end[0], end[0]], [-end[1], end[1]])
        marker.set_xdata([k, k])
        caption.set_text(f"{captions[s]}\nt = {t[k]:.2f},  det = {d[k]:+.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 120), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'matrix': 'tour',
    'a11': 1.5,
    'a12': 0.5,
    'a21': 0.0,
    'a22': -1.0,
    'grid_lines': 17
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.linear_maps_2d")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.linear_maps_2d.main")
problem_dir = Path("problems/maths/linear_maps_2d")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
